In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mis_dro.dataset import *
from mis_dro.plot import *
from mis_dro.newsvendor import newsvendor_cost_cvxpy

In [ ]:
from matplotlib import rc
rc('font', **{'family': 'serif', 'serif': ['Computer Modern'], "size":16})
# rc('text', usetex=True)

In [ ]:
# dgp = "contaminated_normal" #"contaminated_exp" #contaminated_normal
dgps = ["contaminated_normal", "contaminated_exp"]

In [ ]:
df_list = []
# for name in ["cont_normal_9010_mmd", "cont_normal_9010_bas", "bdro_cont_normal_9010"]:
for name in ["kl_cont_exp_9010", "cont_exp_9010_mmd", "cont_normal_9010_mmd", "cont_normal_9010_bas", "bdro_cont_normal_9010"]:
    df_list.append(pd.read_csv(Path(f"/dcs/pg23/u1604520/misdro/results/opt_results/{name}/results.csv")))
all_results_df = pd.concat(df_list)

In [ ]:
all_results_df["num_observations"].unique(), all_results_df["num_posterior_samples"].unique(), all_results_df["num_likelihood_samples"].unique()

In [ ]:
dgp = dgps[0]
results_df1 = preprocess_results_df(all_results_df, dgp)
agg_df1 = get_agg_df(results_df1, ["algorithm", "dgp", "epsilon", "inference", "contamination"]) 
agg_df1.head(3)

dgp = dgps[1]
results_df2 = preprocess_results_df(all_results_df, dgp)
agg_df2 = get_agg_df(results_df2, ["algorithm", "dgp", "epsilon", "inference", "contamination"]) 
agg_df2.head(3)

In [ ]:
df_reset = agg_df1.reset_index()

# Define the filtering conditions for each algorithm
condition = (
#    (df_reset['epsilon'] > 0.3)
    ((df_reset['algorithm'] == 'dro_bas_mmd') & (df_reset['epsilon'] > 0.5)) |
    ((df_reset['algorithm'] == 'empirical_mmd') & (df_reset['epsilon'] > 0.2)) |
       ((df_reset['algorithm'] == 'kl_dro_bas') & (df_reset['epsilon'] > 0.5)) |
   ((df_reset['algorithm'] == 'kl_pp') & (df_reset['epsilon'] > 0.5)) |
    ((df_reset['algorithm'] == 'kl_bdro') & (df_reset['epsilon'] > 0.5))
)

# Negate the condition to keep rows not matching the criteria
df_trimmed = df_reset[~condition]
df_trimmed = df_trimmed.set_index(agg_df1.index.names) 

In [ ]:
df_reset = agg_df2.reset_index()

# Define the filtering conditions for each algorithm
condition = (
   ((df_reset['algorithm'] == 'dro_bas_mmd') & (df_reset['epsilon'] > 0.5)) |
   ((df_reset['algorithm'] == 'empirical_mmd') & (df_reset['epsilon'] > 0.5)) |
   ((df_reset['algorithm'] == 'kl_dro_bas') & (df_reset['epsilon'] > 0.3)) |
   ((df_reset['algorithm'] == 'kl_pp') & (df_reset['epsilon'] > 0.3)) |
    ((df_reset['algorithm'] == 'kl_bdro') & (df_reset['epsilon'] > 0.3))
)

# Negate the condition to keep rows not matching the criteria
df_trimmed2 = df_reset[~condition]
df_trimmed2 = df_trimmed2.set_index(agg_df2.index.names)

In [ ]:
cont_list = agg_df1.index.get_level_values("contamination").unique().tolist()
# dgp_list = agg_df.index.get_level_values("dgp").unique().tolist()
dgp_list = dgps

nrows = len(dgp_list)
ncols = len(cont_list)


fig, axes = plt.subplots(nrows=nrows, ncols=ncols, sharex=False, sharey=False, figsize=(20, 8))
fig.subplots_adjust(wspace=0.2)
pd.options.mode.chained_assignment = None  # default='warn'
# fig.suptitle('Symmetric newsvendor', fontsize=16)
total_samples = 900
for i, dgp in enumerate(dgp_list):
    if i == 0:
        df_trimmed = df_trimmed
    else:
        df_trimmed = df_trimmed2
    for j, cont in enumerate(cont_list):
        axis_df = df_trimmed.loc[:, dgp, :, :, cont]  #agg_df
        axis_df.loc[:, "is_pareto_front"] = is_minimise_pareto_front(axis_df["out_of_sample_var"].values, axis_df["out_of_sample_mean"].values)
        for algorithm in axis_df.index.get_level_values("algorithm").unique():
            if algorithm == 'dro_bas_mmd':
                inference = "npl_mmd"
            elif algorithm in ['kl_bdro', 'kl_dro_bas', 'kl_pp']:
                inference = "bayes"
            else:
                inference = "empirical"
            df = axis_df.loc[algorithm, :, inference]
            alpha = 1.0
            is_labelled=True
            mean_variance_plot(axes[i,j], df, **algorithm_inference_style(algorithm, inference, label_inference=False), offset=0.0, is_labelled=is_labelled, alpha=alpha)
        if j == 0:
            axes[i,j].set_ylabel("Out-of-sample mean", fontsize=16)
        axes[i,j].set_title(f"{NiceNameDGP[dgp]}," + " $\eta$" + f"={cont_list[j]}", fontsize=16) 
        axes[i,j].tick_params(axis='both', which='major', labelsize=16)
        if i == 1:
            axes[i,j].set_xlabel("Out-of-sample variance", fontsize=16)
        if j == 0 and i == 0:
            axes[i,j].legend(fontsize=15)
plt.tight_layout()
fig.savefig("cont.pdf", bbox_inches="tight")

In [ ]:
solve_time_df = get_agg_df(results_df, ["algorithm", "contamination"])
solve_time_df = solve_time_df.round(decimals=4)
solve_time_df["str_solve_time"] = solve_time_df[["mean_solve_time", "std_solve_time"]].apply(
    lambda x: f"{x['mean_solve_time']} ({x['std_solve_time']})", axis=1
)
solve_time_df["str_sample_time"] = solve_time_df[["mean_sample_time", "std_sample_time"]].apply(
    lambda x: f"{x['mean_sample_time']} ({x['std_sample_time']})", axis=1
)
print(solve_time_df[["str_solve_time", "str_sample_time"]].unstack(level="algorithm").to_latex())